# Tablero H&S — carga de datos

Lee la matriz de capacitaciones y escribe las cinco hojas que consume Looker Studio.

**Orden de uso**

1. Ejecutar todas las celdas de definición (Entorno de ejecución → Ejecutar todo).
2. Correr `revisar()`. **Solo lee**: no escribe nada. Comprueba que las cuatro fuentes
   traen datos y muestra qué encontró.
3. Si el reporte se ve bien, correr `cargar_datos()`.

Si `revisar()` marca un problema, `cargar_datos()` se detiene sola antes de tocar el
tablero. Es a propósito: es preferible un tablero viejo a uno vacío.

**De dónde salen los datos**

| Qué | Dónde |
|---|---|
| Matriz de capacitaciones | archivo D6, hoja `Matriz de Capacitaciones H&S` — **solo lectura** |
| Programación, tarifas, presupuesto | `Matriz_HS_Consolidada` |
| Salida | `Datos_tablero_HS` — se reescribe entera |

La matriz se lee **directamente del D6**. No pasa por `IMPORTRANGE` ni por la hoja
`m_cap_H&S`: esa importación superaba el tamaño máximo de un libro de Sheets y por eso
el tablero se quedó sin datos.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CONFIGURACIÓN — lo único que se edita
# ══════════════════════════════════════════════════════════════════════

# --- Origen: archivo D6. Este notebook NUNCA le escribe. -------------
ID_D6            = "1GVq4oN3sn3aomGi_hN_kDOf3Oe4F497Da0l8GX__cjQ"
HOJA_MATRIZ      = "Matriz de Capacitaciones H&S"
FILA_CURSOS      = 6    # fila donde está NOMBRE DEL CURSO
FILA_TITULOS     = 7    # fila de los encabezados
COL_PRIMER_CURSO = 10   # primera columna de cursos (1 = columna A)

# --- Libro de apoyo: programación, tarifas y presupuesto -------------
# Deje ID_APOYO vacío y se abre por nombre. revisar() imprime el ID:
# péguelo aquí para que quede amarrado al archivo exacto.
ID_APOYO          = ""
NOMBRE_APOYO      = "Matriz_HS_Consolidada"
HOJA_PROGRAMACION = "PROGRAMACION"
HOJA_TARIFAS      = "Tarifas"
HOJA_PRESUPUESTO  = "Presupuesto_inicial"

# --- Salida: el libro que lee Looker Studio --------------------------
ID_TABLERO     = ""
NOMBRE_TABLERO = "Datos_tablero_HS"

HOJA_DATOS            = "datos"
HOJA_RESUMEN          = "resumen"
HOJA_RESUMEN_COMPLETO = "resumen_completo"
HOJA_TABLA            = "tabla capacitaciones"
HOJA_PRESUPUESTO_SAL  = "presupuesto"

# --- Reglas de negocio ------------------------------------------------
# Cursos que no entran en los cálculos.
CURSOS_EXCLUIDOS = [
    "EXAMEN MEDICO  OCUPACIONAL",
    "Aparejador/señalero",
    "Polipasto < 5 Toneladas",
    "Puente grua / Polipasto",
]

# Estados que la matriz trae escritos en la columna de fecha.
ESTADOS_TEXTO = {
    "NO APLICA":  "No aplica",
    "PENDIENTE":  "Pendiente",
    "EXCEPTUADO": "Exceptuado",
    "#N/A":       "Sin información",
    "":           "Sin información",
}

# Estados que no cuentan como capacitación por hacer.
ESTADOS_FUERA_DE_RESUMEN = ["Exceptuado", "No aplica", "Vitalicio", "Sin información"]

# Grupos de personal que entran en los resúmenes.
GRUPOS_INCLUIDOS = ["Activos", "Externos"]

# Solo se consideran asistencias desde este año.
ANIO_MINIMO_ASISTENCIA = 2017

# Si una fuente trae menos filas que esto, el proceso se detiene.
MINIMO_FILAS = {
    "matriz":       100,
    "programacion": 100,
    "tarifas":       10,
    "presupuesto":   10,
}


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Conexión y utilidades
# ══════════════════════════════════════════════════════════════════════
from google.colab import auth
from google.auth import default
from datetime import date

import gspread
import numpy as np
import pandas as pd

_cliente = None


def conectar():
    """Autentica una sola vez por sesión y devuelve el cliente de gspread."""
    global _cliente
    if _cliente is None:
        auth.authenticate_user()
        credenciales, _ = default()
        _cliente = gspread.authorize(credenciales)
    return _cliente


def abrir_apoyo():
    """Matriz_HS_Consolidada: programación, tarifas y presupuesto."""
    gc = conectar()
    return gc.open_by_key(ID_APOYO) if ID_APOYO else gc.open(NOMBRE_APOYO)


def abrir_tablero():
    """Datos_tablero_HS: el libro que lee Looker Studio."""
    gc = conectar()
    return gc.open_by_key(ID_TABLERO) if ID_TABLERO else gc.open(NOMBRE_TABLERO)


def normalizar(serie):
    """Texto comparable entre hojas: sin espacio duro, sin sobrantes, en minúscula.

    El espacio duro (\xa0) llega de Sheets y rompe cualquier cruce sin que se vea.
    """
    return (serie.astype(str)
                 .str.replace("\xa0", "", regex=False)
                 .str.strip()
                 .str.lower())


def a_numero(serie):
    """'$1.500.000', '1500000' o 1500000.0 -> número.

    El punto se quita solo cuando separa miles (va seguido de tres dígitos).
    Así '1500000.0' sigue siendo 1.500.000 y no 15.000.000.
    """
    texto = (serie.astype(str)
                  .str.strip()
                  .str.replace("$", "", regex=False)
                  .str.replace(" ", "", regex=False)
                  .str.replace(r"\.(?=\d{3}(\D|$))", "", regex=True)
                  .str.replace(",", ".", regex=False))
    return pd.to_numeric(texto, errors="coerce").fillna(0)


def a_fecha(serie):
    """Lo que sea fecha dd/mm/aaaa queda como fecha; el resto queda vacío (NaT)."""
    limpio = serie.astype(str).str.replace("\xa0", "", regex=False).str.strip()
    return pd.to_datetime(limpio, format="%d/%m/%Y", errors="coerce")


def leer_hoja_df(hoja):
    """DataFrame de una hoja, tolerando encabezados vacíos o repetidos.

    Se usa get_all_values() y no get_all_records() porque este libro tiene
    columnas sin título (la llave de Tarifas, por ejemplo) y eso hace fallar la
    segunda. Todo llega como texto y se convierte donde hace falta.
    """
    filas = hoja.get_all_values()
    if not filas:
        return pd.DataFrame()

    titulos, vistos = [], {}
    for i, bruto in enumerate(filas[0]):
        nombre = bruto.replace("\xa0", "").strip() or f"COLUMNA_{i}"
        if nombre in vistos:
            vistos[nombre] += 1
            nombre = f"{nombre}_{vistos[nombre]}"
        else:
            vistos[nombre] = 0
        titulos.append(nombre)

    ancho = len(titulos)
    cuerpo = [f + [""] * (ancho - len(f)) if len(f) < ancho else f[:ancho]
              for f in filas[1:]]
    return pd.DataFrame(cuerpo, columns=titulos)


class FuenteVacia(RuntimeError):
    """Una hoja de origen no trajo datos suficientes. El proceso se detiene."""


def exigir_filas(df, clave, nombre_hoja):
    minimo = MINIMO_FILAS[clave]
    if len(df) < minimo:
        raise FuenteVacia(
            f"La hoja {nombre_hoja!r} trajo {len(df)} filas y se esperaban al menos "
            f"{minimo}. No se escribió nada en el tablero.\n"
            "Revise que la hoja de origen tenga datos antes de volver a ejecutar."
        )
    return df


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  1. Leer la matriz del D6 y desplegarla a formato largo
# ══════════════════════════════════════════════════════════════════════

def leer_matriz():
    """Devuelve (df, info) con una fila por persona y curso.

    Los cursos NO se ubican por posición fija: se recorre la fila de nombres de
    curso y, dentro del bloque de cada uno, se busca su columna de fecha y la de
    soporte por el título. Así el notebook sigue funcionando si mañana un curso
    ocupa dos columnas en vez de tres, o si se agregan cursos nuevos.
    """
    hoja = conectar().open_by_key(ID_D6).worksheet(HOJA_MATRIZ)
    filas = hoja.get_all_values()

    if len(filas) <= FILA_TITULOS:
        raise FuenteVacia(
            f"{HOJA_MATRIZ!r} trajo {len(filas)} filas: no hay datos debajo de los "
            f"encabezados (fila {FILA_TITULOS}). No se escribió nada."
        )

    fila_cursos  = filas[FILA_CURSOS - 1]
    fila_titulos = filas[FILA_TITULOS - 1]
    cuerpo       = filas[FILA_TITULOS:]

    def indice(titulo):
        buscado = titulo.strip().upper()
        for i, valor in enumerate(fila_titulos):
            if valor.strip().upper() == buscado:
                return i
        return None

    # Columnas que identifican a la persona
    obligatorias = ["STATUS DEL PERSONAL", "CODIGO", "ID",
                    "DIVISIÓN", "NOMBRE", "POSICIÓN"]
    fijas, faltantes = {}, []
    for titulo in obligatorias:
        i = indice(titulo)
        if i is None:
            faltantes.append(titulo)
        else:
            fijas[titulo] = i
    if faltantes:
        raise FuenteVacia(
            f"No encontré estas columnas en la fila {FILA_TITULOS} de {HOJA_MATRIZ!r}: "
            f"{faltantes}. Si las renombraron, ajuste la lista 'obligatorias'."
        )

    # 'Grupo de personal' alimenta los dos resúmenes. Si no está, se avisa.
    i_grupo = indice("Grupo de personal")

    # Un bloque por curso: nombre en FILA_CURSOS, fecha y soporte por título
    inicios = [i for i, v in enumerate(fila_cursos)
               if i >= COL_PRIMER_CURSO - 1 and v.strip()]
    bloques = []
    for n, inicio in enumerate(inicios):
        fin = inicios[n + 1] if n + 1 < len(inicios) else len(fila_titulos)
        i_fecha = i_soporte = None
        for j in range(inicio, min(fin, len(fila_titulos))):
            titulo = fila_titulos[j].strip().upper()
            if i_fecha is None and "FECHA" in titulo:
                i_fecha = j
            if i_soporte is None and "SOPORTE" in titulo:
                i_soporte = j
        bloques.append({"curso": fila_cursos[inicio].strip(),
                        "fecha": i_fecha,
                        "soporte": i_soporte})

    def celda(fila, i):
        return fila[i] if (i is not None and i < len(fila)) else ""

    registros, personas = [], 0
    for fila in cuerpo:
        if not celda(fila, fijas["NOMBRE"]).strip():
            continue
        if celda(fila, fijas["STATUS DEL PERSONAL"]).strip().upper() != "ACTIVO":
            continue
        personas += 1
        base = {
            "CODIGO ":  celda(fila, fijas["CODIGO"]),
            "ID":       celda(fila, fijas["ID"]),
            "DIVISIÓN": celda(fila, fijas["DIVISIÓN"]),
            "NOMBRE":   celda(fila, fijas["NOMBRE"]),
            "POSICIÓN": celda(fila, fijas["POSICIÓN"]),
        }
        if i_grupo is not None:
            base["Grupo de personal"] = celda(fila, i_grupo)
        for bloque in bloques:
            registros.append({**base,
                              "CURSO/CAPACITACION": bloque["curso"],
                              "Fecha_texto": celda(fila, bloque["fecha"]),
                              "Soporte":     celda(fila, bloque["soporte"])})

    df = pd.DataFrame(registros)
    info = {
        "cursos":           len(bloques),
        "cursos_sin_fecha": [b["curso"] for b in bloques if b["fecha"] is None],
        "personas":         personas,
        "filas":            len(df),
        "tiene_grupo":      i_grupo is not None,
    }
    exigir_filas(df, "matriz", HOJA_MATRIZ)
    return df, info


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  2. Estados y vigencias
# ══════════════════════════════════════════════════════════════════════

def preparar_matriz(df):
    """Convierte el texto de la matriz en ESTADO y Fecha.

    La columna de vencimiento mezcla fechas reales con palabras (NO APLICA,
    PENDIENTE, EXCEPTUADO). Lo que sea fecha se calcula contra hoy; lo demás se
    toma tal como viene escrito en la hoja.
    """
    hoy = pd.Timestamp(date.today())
    df = df.copy()

    texto = (df["Fecha_texto"].astype(str)
               .str.replace("\xa0", "", regex=False)
               .str.strip())

    df["Fecha"]  = pd.to_datetime(texto, format="%d/%m/%Y", errors="coerce")
    df["ESTADO"] = texto.str.upper().map(ESTADOS_TEXTO)

    # Fechas comodín: cualquier año superior a 2100 es "no vence nunca".
    # Se evalúa por el año y no contra una lista de fechas concretas, para que
    # un comodín nuevo (01/01/9999, 07/05/9998…) no se cuele como fecha real.
    vitalicio = df["Fecha"].dt.year > 2100
    df.loc[vitalicio, "ESTADO"] = "Vitalicio"
    df.loc[vitalicio, "Fecha"]  = pd.NaT

    con_fecha = df["Fecha"].notna()
    df.loc[con_fecha & (df["Fecha"] >= hoy), "ESTADO"] = "Vigente"
    df.loc[con_fecha & (df["Fecha"] <  hoy), "ESTADO"] = "Vencido"

    # Nada se pierde en silencio: lo que no se reconoce queda marcado y se puede
    # contar desde el tablero.
    df["ESTADO"] = df["ESTADO"].fillna("Sin clasificar")
    df["estado_definitivo"] = df["ESTADO"]
    df["Soporte"] = df["Soporte"].astype(str).str.strip().str.capitalize()

    excluidos = [c.strip().lower() for c in CURSOS_EXCLUIDOS]
    df = df[~normalizar(df["CURSO/CAPACITACION"]).isin(excluidos)]

    return df.reset_index(drop=True)


def hoja_datos(df):
    """Las columnas de la hoja 'datos', en el mismo orden de siempre."""
    return df[["CODIGO ", "ID", "DIVISIÓN", "NOMBRE", "POSICIÓN", "ESTADO",
               "CURSO/CAPACITACION", "Fecha", "estado_definitivo", "Soporte"]]


def a_capacitar(df):
    """Cuántas personas hay que capacitar por año, mes, división y curso.

    Lo que está Pendiente se cuenta en el mes en curso; lo demás, en el mes en
    que vence.
    """
    hoy = date.today()
    d = df.copy()
    d["año_vencimiento"] = np.where(d["estado_definitivo"] == "Pendiente",
                                    hoy.year, d["Fecha"].dt.year)
    d["mes_vencimiento"] = np.where(d["estado_definitivo"] == "Pendiente",
                                    hoy.month, d["Fecha"].dt.month)
    return (d.groupby(["año_vencimiento", "mes_vencimiento",
                       "DIVISIÓN", "CURSO/CAPACITACION"])["CODIGO "]
             .count().reset_index())


def resumenes(df, tarifas):
    """Los dos resúmenes: por curso, y con año, mes y costo."""
    if "Grupo de personal" not in df.columns:
        raise FuenteVacia(
            "La matriz no trae la columna 'Grupo de personal', que es la que separa "
            "Activos de Externos en los dos resúmenes. Ubique en qué columna quedó y "
            "agréguela a leer_matriz(). No se escribió nada."
        )

    hoy = pd.Timestamp(date.today())
    anio_proximo = date.today().year + 1

    d = df[(df["Fecha"].dt.year < anio_proximo) | (df["Fecha"].isna())]
    d = d[~d["ESTADO"].isin(ESTADOS_FUERA_DE_RESUMEN)]
    d = d[d["Grupo de personal"].isin(GRUPOS_INCLUIDOS)].copy()
    d["CURSO/CAPACITACION"] = d["CURSO/CAPACITACION"].str.strip()

    resumen = (d.groupby(["CURSO/CAPACITACION", "Grupo de personal", "DIVISIÓN"])["NOMBRE"]
                .count().reset_index(name="COUNT"))

    con_mes = d.copy()
    con_mes["Fecha"] = con_mes["Fecha"].fillna(hoy)
    con_mes["AÑO"] = con_mes["Fecha"].dt.year
    con_mes["MES"] = con_mes["Fecha"].dt.month
    completo = (con_mes.groupby(["CURSO/CAPACITACION", "Grupo de personal",
                                 "DIVISIÓN", "AÑO", "MES"], dropna=False)["NOMBRE"]
                       .count().reset_index(name="COUNT"))

    # La llave se arma normalizada y con separador: sin eso, un espacio de más
    # rompe el cruce y el costo queda en cero sin que nadie lo note.
    completo["LLAVE"] = (normalizar(completo["CURSO/CAPACITACION"]) + "|" +
                         normalizar(completo["Grupo de personal"]) + "|" +
                         normalizar(completo["DIVISIÓN"]))
    completo = completo.merge(tarifas[["LLAVE", "COSTO CAPACITACION"]],
                              on="LLAVE", how="left")

    sin_tarifa = int(completo["COSTO CAPACITACION"].isna().sum())
    completo["COSTO CAPACITACION"] = completo["COSTO CAPACITACION"].fillna(0).astype(int)
    completo["Fecha_completa"] = pd.to_datetime(
        completo["AÑO"].astype(str) + "-" + completo["MES"].astype(str) + "-01",
        errors="coerce")

    return resumen, completo, sin_tarifa


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3. Programación, tarifas y presupuesto (Matriz_HS_Consolidada)
# ══════════════════════════════════════════════════════════════════════

VACIOS = {"": "SIN INFORMACIÓN", "#N/A": "SIN INFORMACIÓN", "#VALUE!": "SIN INFORMACIÓN"}


def leer_tarifas():
    """Hoja Tarifas, con la llave armada igual que en los resúmenes."""
    df = leer_hoja_df(abrir_apoyo().worksheet(HOJA_TARIFAS))
    exigir_filas(df, "tarifas", HOJA_TARIFAS)

    df = df.rename(columns={"CAPACITACION": "CAPACITACIONES"})
    df["COSTO CAPACITACION"] = a_numero(df["COSTO CAPACITACION"]).astype(int)
    df["LLAVE"] = (normalizar(df["CAPACITACIONES"]) + "|" +
                   normalizar(df["Grupo de personal"]) + "|" +
                   normalizar(df["DIVISION"]))
    return df.drop_duplicates(subset=["LLAVE"], keep="last")


def leer_programacion():
    """Hoja PROGRAMACION, limpia y con año y mes de asistencia."""
    df = leer_hoja_df(abrir_apoyo().worksheet(HOJA_PROGRAMACION))
    exigir_filas(df, "programacion", HOJA_PROGRAMACION)

    df = df[df["IDREGISTRO"].astype(str).str.strip() != ""].copy()

    for columna in ["NOMBRE", "DIVISIÓN", "CAPACITACIONES", "FECHA DE VENCIMIENTO",
                    "PROVEEDOR", "ID", "POSICIÓN"]:
        if columna in df.columns:
            df[columna] = df[columna].astype(str).str.strip().replace(VACIOS)

    for columna in ["ASISTENCIA", "CODIGO "]:
        if columna in df.columns:
            df[columna] = (df[columna].astype(str).str.strip()
                             .replace({"": "POR CONFIRMAR"}))

    if "VIGENCIA" in df.columns:
        df["VIGENCIA"] = pd.to_numeric(df["VIGENCIA"], errors="coerce").fillna(0)

    df["FECHA DE ASISTENCIA"] = a_fecha(df["FECHA DE ASISTENCIA"])
    df["año_asistencia"] = df["FECHA DE ASISTENCIA"].dt.year
    df["mes_asistencia"] = df["FECHA DE ASISTENCIA"].dt.month
    return df


def cronograma_capacitaciones():
    """Programados por año/mes/división/curso, y lo mismo abierto por asistencia."""
    df = leer_programacion()
    sin_fecha = int(df["FECHA DE ASISTENCIA"].isna().sum())

    df = df[df["año_asistencia"] > ANIO_MINIMO_ASISTENCIA]

    programados = (df.groupby(["año_asistencia", "mes_asistencia",
                               "DIVISIÓN", "CAPACITACIONES"])["IDREGISTRO"]
                     .count().reset_index())
    por_asistencia = (df.groupby(["año_asistencia", "mes_asistencia", "DIVISIÓN",
                                  "CAPACITACIONES", "ASISTENCIA"])["IDREGISTRO"]
                        .count().reset_index())
    return programados, por_asistencia, sin_fecha


def tabla_capacitaciones(pendientes, programados, por_asistencia):
    """Une lo que hay que capacitar con lo programado y lo asistido."""
    pendientes = pendientes.rename(columns={
        "año_vencimiento": "año", "mes_vencimiento": "mes", "DIVISIÓN": "Division",
        "CURSO/CAPACITACION": "Capacitaciones", "CODIGO ": "A_capacitar"})
    renombre = {"año_asistencia": "año", "mes_asistencia": "mes", "DIVISIÓN": "Division",
                "CAPACITACIONES": "Capacitaciones", "IDREGISTRO": "Programados"}
    programados    = programados.rename(columns=renombre)
    por_asistencia = por_asistencia.rename(columns=renombre)

    for tabla in (pendientes, programados, por_asistencia):
        tabla["Division"]       = normalizar(tabla["Division"])
        tabla["Capacitaciones"] = normalizar(tabla["Capacitaciones"])

    llaves = ["año", "mes", "Division", "Capacitaciones"]
    total = pd.concat([pendientes, programados])
    total = total.groupby(llaves)[["A_capacitar", "Programados"]].sum().reset_index()
    total["Total"] = total.A_capacitar + total.Programados

    asistio  = por_asistencia[por_asistencia.ASISTENCIA == "SI"].copy()
    no_fue   = por_asistencia[por_asistencia.ASISTENCIA == "NO"].copy()
    en_duda  = por_asistencia[por_asistencia.ASISTENCIA.isin(
                   ["POR CONFIRMAR", "SIN INFORMACIÓN"])].copy()
    otros    = por_asistencia[~por_asistencia.ASISTENCIA.isin(
                   ["SI", "NO", "POR CONFIRMAR", "SIN INFORMACIÓN"])]

    asistio  = asistio.rename(columns={"Programados": "Asistieron"})
    no_fue   = no_fue.rename(columns={"Programados": "No_Asistieron"})
    # El nombre de esta columna va con la errata original: el tablero de Looker
    # ya tiene campos apuntando a "Pendiente_confimacion".
    en_duda  = en_duda.rename(columns={"Programados": "Pendiente_confimacion"})

    columnas = ["A_capacitar", "Programados", "Total"]
    for parte, nueva in ((asistio, "Asistieron"),
                         (no_fue, "No_Asistieron"),
                         (en_duda, "Pendiente_confimacion")):
        columnas.append(nueva)
        total = pd.concat([total, parte])
        total = total.groupby(llaves)[columnas].sum().reset_index()

    total["Pendiente_programar"] = total.Total - total.Asistieron
    return total, int(len(otros))


def seguimiento_presupuesto(tarifas):
    """Presupuesto inicial contra lo ejecutado en el año en curso."""
    libro = abrir_apoyo()

    inicial = leer_hoja_df(libro.worksheet(HOJA_PRESUPUESTO))
    exigir_filas(inicial, "presupuesto", HOJA_PRESUPUESTO)
    inicial["total_a_capacitar"] = a_numero(inicial["total_a_capacitar"])
    inicial["VALOR"] = a_numero(inicial["VALOR"])
    inicial["AÑO"] = pd.to_numeric(inicial["AÑO"], errors="coerce")
    inicial["MES"] = pd.to_numeric(inicial["MES"], errors="coerce")

    ejecutado = leer_programacion()
    ejecutado = ejecutado[ejecutado["NOMBRE"].astype(str).str.strip() != ""]
    ejecutado = ejecutado[ejecutado["ASISTENCIA"] == "SI"].copy()
    ejecutado["AÑO"] = ejecutado["FECHA DE ASISTENCIA"].dt.year
    ejecutado["MES"] = ejecutado["FECHA DE ASISTENCIA"].dt.month
    # El año sale del reloj, no está escrito en el código: en enero seguía
    # mostrando la ejecución del año anterior.
    ejecutado = ejecutado[ejecutado["AÑO"] == date.today().year]

    for tabla in (inicial, ejecutado):
        tabla["DIVISIÓN"]       = normalizar(tabla["DIVISIÓN"])
        tabla["CAPACITACIONES"] = normalizar(tabla["CAPACITACIONES"])

    llaves = ["AÑO", "MES", "DIVISIÓN", "CAPACITACIONES"]

    presupuestado = (inicial.groupby(llaves)
                            .agg(total_a_capacitar=("total_a_capacitar", "sum"),
                                 VALOR=("VALOR", "max"))
                            .reset_index())
    realizado = (ejecutado.groupby(llaves)
                          .agg(capacitados=("IDREGISTRO", "count"))
                          .reset_index())

    # Un merge y no un concat: así presupuesto y ejecución quedan en la MISMA
    # fila y el tablero puede compararlos sin sumar por su cuenta.
    df = presupuestado.merge(realizado, on=llaves, how="outer")
    df["total_a_capacitar"] = df["total_a_capacitar"].fillna(0)
    df["capacitados"]       = df["capacitados"].fillna(0).astype(int)
    df["VALOR"]             = a_numero(df["VALOR"])

    df = df.rename(columns={"DIVISIÓN": "DIVISION"})
    df["LLAVE"] = normalizar(df["CAPACITACIONES"]) + "|" + normalizar(df["DIVISION"])

    costos = tarifas.copy()
    costos["LLAVE"] = normalizar(costos["CAPACITACIONES"]) + "|" + normalizar(costos["DIVISION"])
    costos = costos.drop_duplicates(subset=["LLAVE"], keep="last")

    df = df.merge(costos[["LLAVE", "COSTO CAPACITACION"]], on="LLAVE", how="left")
    sin_tarifa = int(df["COSTO CAPACITACION"].isna().sum())
    df["COSTO CAPACITACION"] = df["COSTO CAPACITACION"].fillna(0).astype(int)

    df["Valor_capacitados"]  = df["COSTO CAPACITACION"] * df["capacitados"]
    df["Valor_presupuesto"]  = df["VALOR"] * df["total_a_capacitar"]

    excluidos = [c.strip().lower() for c in CURSOS_EXCLUIDOS]
    df = df[~df["CAPACITACIONES"].isin(excluidos)]

    return df.drop(columns=["LLAVE"]), sin_tarifa


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  4. Escritura en el tablero
# ══════════════════════════════════════════════════════════════════════

def _filas_para_sheets(df):
    """Convierte el DataFrame a filas.

    Las fechas salen como dd/mm/aaaa y los números como números. El código
    anterior mandaba todo con astype(str), y por eso a Looker le llegaban las
    fechas como texto ('2026-05-13 00:00:00') y no se podían graficar.
    """
    d = df.copy()
    for columna in d.columns:
        if pd.api.types.is_datetime64_any_dtype(d[columna]):
            d[columna] = d[columna].dt.strftime("%d/%m/%Y")

    filas = []
    for registro in d.itertuples(index=False, name=None):
        fila = []
        for valor in registro:
            if valor is None or (isinstance(valor, float) and pd.isna(valor)):
                fila.append("")
            elif hasattr(valor, "item"):      # numpy int64, float64…
                fila.append(valor.item())
            else:
                fila.append(valor)
        filas.append(fila)
    return filas


def escribir_hoja(libro, nombre, df):
    """Reemplaza el contenido de una hoja por el DataFrame."""
    hoja = libro.worksheet(nombre)
    hoja.clear()
    hoja.update([df.columns.tolist()] + _filas_para_sheets(df),
                value_input_option="USER_ENTERED")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  5. Revisión y carga
# ══════════════════════════════════════════════════════════════════════

def calcular():
    """Arma las cinco tablas del tablero. No escribe nada."""
    matriz, info = leer_matriz()
    matriz = preparar_matriz(matriz)

    tarifas = leer_tarifas()
    datos = hoja_datos(matriz)
    resumen, resumen_completo, sin_tarifa_resumen = resumenes(matriz, tarifas)

    programados, por_asistencia, sin_fecha = cronograma_capacitaciones()
    tabla, asistencias_raras = tabla_capacitaciones(
        a_capacitar(matriz), programados, por_asistencia)

    presupuesto, sin_tarifa_presupuesto = seguimiento_presupuesto(tarifas)

    tablas = {
        HOJA_DATOS:            datos,
        HOJA_RESUMEN:          resumen,
        HOJA_RESUMEN_COMPLETO: resumen_completo,
        HOJA_TABLA:            tabla,
        HOJA_PRESUPUESTO_SAL:  presupuesto,
    }
    info.update({
        "estados":                     matriz["ESTADO"].value_counts().to_dict(),
        "programacion_sin_fecha":      sin_fecha,
        "asistencias_no_reconocidas":  asistencias_raras,
        "resumen_sin_tarifa":          sin_tarifa_resumen,
        "presupuesto_sin_tarifa":      sin_tarifa_presupuesto,
    })
    return tablas, info


def revisar():
    """Calcula todo y reporta, SIN escribir en el tablero.

    Correr esto antes de cargar_datos() la primera vez, y cada vez que alguien
    haya movido algo en las hojas de origen.
    """
    tablas, info = calcular()

    print("MATRIZ (D6)")
    print(f"  cursos detectados ....... {info['cursos']}")
    print(f"  personas activas ........ {info['personas']}")
    print(f"  filas persona x curso ... {info['filas']}")
    if info["cursos_sin_fecha"]:
        print(f"  ⚠ cursos sin columna de fecha: {info['cursos_sin_fecha']}")
    if not info["tiene_grupo"]:
        print("  ⚠ no se encontró la columna 'Grupo de personal'")

    print()
    print("ESTADOS")
    for estado, cantidad in sorted(info["estados"].items(),
                                   key=lambda x: -x[1]):
        marca = " ⚠" if estado == "Sin clasificar" else ""
        print(f"  {estado:<18} {cantidad:>8}{marca}")

    print()
    print("AVISOS")
    avisos = [
        ("programación sin fecha de asistencia (no entra al tablero)",
         info["programacion_sin_fecha"]),
        ("valores de ASISTENCIA no reconocidos", info["asistencias_no_reconocidas"]),
        ("filas del resumen sin tarifa (costo queda en 0)", info["resumen_sin_tarifa"]),
        ("filas del presupuesto sin tarifa", info["presupuesto_sin_tarifa"]),
    ]
    for texto, cantidad in avisos:
        print(f"  {'⚠' if cantidad else ' '} {texto}: {cantidad}")

    print()
    print("HOJAS QUE SE ESCRIBIRÍAN")
    for nombre, df in tablas.items():
        print(f"  {nombre:<22} {len(df):>7} filas   {list(df.columns)}")

    print()
    print("No se escribió nada. Si esto se ve bien, ejecute cargar_datos().")
    return tablas, info


def cargar_datos():
    """Calcula todo y luego reescribe las cinco hojas del tablero.

    Primero se calcula y solo después se toca el libro de salida: si algo falla,
    el tablero se queda como estaba en vez de quedar a medio escribir.
    """
    tablas, info = calcular()

    libro = abrir_tablero()
    for nombre, df in tablas.items():
        escribir_hoja(libro, nombre, df)
        print(f"  escrita {nombre:<22} {len(df):>7} filas")

    print()
    print("Listo. Revise en Drive que", NOMBRE_TABLERO,
          "quede con la fecha de hoy, y actualice los datos en Looker Studio.")
    return info


In [ ]:
# Revisión: solo lee, no escribe nada.
tablas, info = revisar()


In [ ]:
# Escritura real del tablero. Correr solo si revisar() se vio bien.
cargar_datos()
